In [ ]:
import numpy as np
from pathlib import Path
import h5py
from plot import show_diffractograms
from concurrent.futures import ProcessPoolExecutor, as_completed
from create_targets_fn import enhance_single

dataset_dir = Path("dataset/")
split_name = "train"
result_path = dataset_dir / f"{split_name}_target.h5"

In [ ]:
dataset = h5py.File(dataset_dir / f"{split_name}.h5", 'r')

In [ ]:
skip_rest = False
save_data = True
scale_factor = 1
show_n = 2

In [ ]:
# Create file
if save_data:
    with h5py.File(result_path, 'w') as f:
        pass

def save_dataset(n, d):
    if not save_data:
        return
    
    with h5py.File(result_path, 'r+') as f:
        chunk_shape = (1, *d.shape[1:])
        f.create_dataset(
                n, 
                data=d,  
                compression="lzf", 
                chunks=chunk_shape,
                shuffle=True 
            )

In [ ]:
def enhance_fn_mc(dataset, sigma, thr, area_size, normalize, show_n=0, skip_rest=False):
    imgs = [None] * len(dataset)

    with ProcessPoolExecutor() as executor:
        futures = {executor.submit(enhance_single, dataset[i], sigma, thr, area_size, normalize, scale_factor): i 
                   for i in range(len(dataset)) if not skip_rest or i < show_n}
        
        for future in as_completed(futures):
            i = futures[future]
            imgs[i] = future.result()
        
    for i in range(show_n):
        show_diffractograms({
            "Before": dataset[i],
            "After": imgs[i]
        }, title=i)

    if not skip_rest:
        print(len(imgs))
        return np.stack(imgs)

In [ ]:
name = "au"
result = enhance_fn_mc(dataset[name], sigma=1.5, thr=1.5, area_size=6, normalize=True, show_n=show_n, skip_rest=skip_rest)
save_dataset(name, result)
print(f"{name}: {result.shape}")
result = None # Free memory

In [ ]:
name = "tbf3"
result = enhance_fn_mc(dataset[name], sigma=2.5, thr=1.5, area_size=4, normalize=True, show_n=show_n, skip_rest=skip_rest)
save_dataset(name, result)
print(f"{name}: {result.shape}")
result = None # Free memory

In [ ]:
name = "feo"
result = enhance_fn_mc(dataset[name], sigma=2.5, thr=2, area_size=10, normalize=True, show_n=show_n, skip_rest=skip_rest)
save_dataset(name, result)
print(f"{name}: {result.shape}")
result = None # Free memory

In [ ]:
name = "laf3"
result = enhance_fn_mc(dataset[name], sigma=3, thr=5, area_size=8, normalize=True, show_n=show_n, skip_rest=skip_rest)
save_dataset(name, result)
print(f"{name}: {result.shape}")
result = None # Free memory

In [ ]:
name = "gdf3"
result = enhance_fn_mc(dataset[name], sigma=1.5, thr=1, area_size=8, normalize=True, show_n=show_n, skip_rest=skip_rest)
save_dataset(name, result)
print(f"{name}: {result.shape}")
result = None # Free memory